In [3]:
import os

In [4]:
%pwd

'/Users/ntchindagiscard/Documents/end-end-mlflow/research'

In [5]:
os.chdir("../")

In [6]:
%pwd

'/Users/ntchindagiscard/Documents/end-end-mlflow'

In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataProcessingConfig:
    root_dir: Path
    movies: Path
    ratings: Path
    tags: Path

In [8]:
from mlProject.utils.common import read_yaml, create_directories
from mlProject.constants import *

In [9]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH,
            schema_filepath = SCHEMA_FILE_PATH
            ) -> None:
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_processing_config(self) -> DataTransformationConfig:

        config  = self.config.data_processing

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            movies= config.movies,
            tags= config.tags,
            ratings= config.ratings,
            root_dir= config.root_dir
        )

        return data_transformation_config

In [6]:
import pandas as pd

config = ConfigurationManager()
data_processing_config = config.get_data_processing_config()

movies_df = pd.read_csv(data_processing_config.movies)
ratings_df = pd.read_csv(data_processing_config.ratings)
tags_df = pd.read_csv(data_processing_config.tags)

FileNotFoundError: [Errno 2] No such file or directory: 'config/config.yaml'

In [12]:
movies_df

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


In [13]:
ratings_df

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [14]:
tags_df

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200
...,...,...,...,...
3678,606,7382,for katie,1171234019
3679,606,7936,austere,1173392334
3680,610,3265,gun fu,1493843984
3681,610,3265,heroic bloodshed,1493843978


# Movie Features


## Step 1: One-shot Encode Genre


In [12]:
movies_df['genres'] = movies_df['genres'].str.split('|')
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]


In [13]:
genre_set = set(genre for genres in movies_df['genres'] for genre in genres)

In [14]:
movies_df['genres']

0       [Adventure, Animation, Children, Comedy, Fantasy]
1                          [Adventure, Children, Fantasy]
2                                       [Comedy, Romance]
3                                [Comedy, Drama, Romance]
4                                                [Comedy]
                              ...                        
9737                 [Action, Animation, Comedy, Fantasy]
9738                         [Animation, Comedy, Fantasy]
9739                                              [Drama]
9740                                  [Action, Animation]
9741                                             [Comedy]
Name: genres, Length: 9742, dtype: object

In [15]:
genre_set

{'(no genres listed)',
 'Action',
 'Adventure',
 'Animation',
 'Children',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Fantasy',
 'Film-Noir',
 'Horror',
 'IMAX',
 'Musical',
 'Mystery',
 'Romance',
 'Sci-Fi',
 'Thriller',
 'War',
 'Western'}

In [16]:
for genre in genre_set:
    movies_df[genre] = movies_df['genres'].apply(lambda x: int(genre in x))

In [17]:
movies_df.head()

,movieId,title,genres,Western,Romance,Fantasy,Adventure,Action,Film-Noir,(no genres listed),...,Sci-Fi,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",0,0,1,1,0,0,0,...,0,0,0,1,0,1,0,0,0,1
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",0,0,1,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
4,5,Father of the Bride Part II (1995),[Comedy],0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0


In [18]:
movies_features = movies_df.drop(['genres', 'title'], axis=1)
print(movies_features.count())
movies_features.head()

movieId               9742
Western               9742
Romance               9742
Fantasy               9742
Adventure             9742
Action                9742
Film-Noir             9742
(no genres listed)    9742
IMAX                  9742
War                   9742
Crime                 9742
Sci-Fi                9742
Mystery               9742
Thriller              9742
Comedy                9742
Horror                9742
Children              9742
Musical               9742
Drama                 9742
Documentary           9742
Animation             9742
dtype: int64


,movieId,Western,Romance,Fantasy,Adventure,Action,Film-Noir,(no genres listed),IMAX,War,...,Sci-Fi,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation
0,1,0,0,1,1,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,1
1,2,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,3,0,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,4,0,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
4,5,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0


## Step 2: Combine with tags


In [19]:
movie_tags = tags_df.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
movie_tags

,movieId,tag
0,1,pixar pixar fun
1,2,fantasy magic board game Robin Williams game
2,3,moldy old
3,5,pregnancy remake
4,7,remake
...,...,...
1567,183611,Comedy funny Rachel McAdams
1568,184471,adventure Alicia Vikander video game adaptation
1569,187593,Josh Brolin Ryan Reynolds sarcasm
1570,187595,Emilia Clarke star wars


In [20]:
movies_features = movies_features.merge(movie_tags, on='movieId', how='left')
movies_features['tag'] = movies_features['tag'].fillna('')
movies_features

,movieId,Western,Romance,Fantasy,Adventure,Action,Film-Noir,(no genres listed),IMAX,War,...,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation,tag
0,1,0,0,1,1,0,0,0,0,0,...,0,0,1,0,1,0,0,0,1,pixar pixar fun
1,2,0,0,1,1,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,fantasy magic board game Robin Williams game
2,3,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,moldy old
3,4,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,
4,5,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,pregnancy remake
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9737,193581,0,0,1,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,1,
9738,193583,0,0,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,1,
9739,193585,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,
9740,193587,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=50)
tag_features = tfidf.fit_transform(movies_features['tag']).toarray()

tag_features_names = [f"tag_{i}" for i in range(tag_features.shape[1])]
tag_feature_df = pd.DataFrame(tag_features, columns=tag_features_names)
tag_feature_df.head()

,tag_0,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,tag_9,...,tag_40,tag_41,tag_42,tag_43,tag_44,tag_45,tag_46,tag_47,tag_48,tag_49
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
movies_features = pd.concat([movies_features.reset_index(drop=True), tag_feature_df], axis=1)

movies_features = movies_features.drop(['tag'], axis=1)
print(movies_features.count())
movies_features.head()

movieId      9742
Western      9742
Romance      9742
Fantasy      9742
Adventure    9742
             ... 
tag_45       9742
tag_46       9742
tag_47       9742
tag_48       9742
tag_49       9742
Length: 71, dtype: int64


,movieId,Western,Romance,Fantasy,Adventure,Action,Film-Noir,(no genres listed),IMAX,War,...,tag_40,tag_41,tag_42,tag_43,tag_44,tag_45,tag_46,tag_47,tag_48,tag_49
0,1,0,0,1,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,0,0,1,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0,1,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0,1,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## OOP


In [10]:
# base feature transformer
from abc import ABC, abstractmethod
import pandas as pd

class FeatureExtractor(ABC):

    @abstractmethod
    def generate_features(self) -> pd.DataFrame:
        pass

In [60]:
from sklearn.feature_extraction.text import TfidfVectorizer
from pathlib import Path

class MovieFeatureExtractor(FeatureExtractor):  

    def load_dataset(self, file_path: Path) -> pd.DataFrame:
        return pd.read_csv(file_path)  

    def generate_features(self, movies_path: Path, tags_path: Path) -> pd.DataFrame:
        movies = self.load_dataset(movies_path)
        tags = self.load_dataset(tags_path)
        movies['genres'] = movies['genres'].str.split('|')  
        genre_set = set(genre for genres in movies['genres'] for genre in genres)

        for genre in genre_set:
            movies[genre] = movies['genres'].apply(lambda x: int(genre in x))

        tags_aggregated = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()

        movies_features = movies.merge(tags_aggregated, on='movieId', how='left')
        movies_features['tag'] = movies_features['tag'].fillna('')

        tfidf = TfidfVectorizer(max_features=50)  
        tfidf_features = tfidf.fit_transform(movies_features['tag']).toarray()

        tfidf_df = pd.DataFrame(tfidf_features, columns=[f'tag_{i}' for i in range(tfidf_features.shape[1])])
        movies_features = pd.concat([movies_features.reset_index(drop=True), tfidf_df], axis=1)

        movies_features = movies_features.drop(['genres', 'title', 'tag'], axis=1)

        return movies_features

In [67]:
class UserFeatureExtractor(FeatureExtractor):
    """
    Extracts user features from ratings and movies data.

    Attributes:
        None

    Methods:
        generate_features: Generates user features based on ratings and movies data.

    """
    def load_dataset(self, file_path: Path) -> pd.DataFrame:
        return pd.read_csv(file_path)

    def generate_features(self, ratings_path: Path, movies_path: Path) -> pd.DataFrame:
        """
        Generates user features based on ratings and movies data.

        Args:
            ratings (pd.DataFrame): DataFrame containing user ratings data.
            movies (pd.DataFrame): DataFrame containing movies data.

        Returns:
            pd.DataFrame: DataFrame containing user features.

        """
        ratings = self.load_dataset(ratings_path)
        movies = self.load_dataset(movies_path)
        user_ratings = ratings.groupby('userId')['rating'].agg(['mean', 'count']).reset_index()
        user_ratings.rename(columns={'mean': 'avg_rating', 'count': 'rating_count'}, inplace=True)

        user_genres = ratings.merge(movies[['movieId', 'genres']], on='movieId', how='left')

        user_genres['genres'] = user_genres['genres'].fillna('').str.split('|')
        genre_set = set(genre for genres in user_genres['genres'] for genre in genres)
        
        for genre in genre_set:
            user_genres[genre] = user_genres['genres'].apply(lambda x: int(genre in x))

        user_genre_preferences = user_genres.groupby('userId')[list(genre_set)].mean().reset_index()

        user_features = user_ratings.merge(user_genre_preferences, on='userId', how='left')

        return user_genres

In [80]:
from typing import Tuple
from mlProject import logger
class DataPreprocessor:
    def __init__(self, config: DataTransformationConfig) -> None:
        self.movie_feature_extractor = MovieFeatureExtractor()
        self.user_feature_extractor = UserFeatureExtractor()
        self.config = config
    
    def load_dataset(self, file_path: Path) -> pd.DataFrame:
        return pd.read_csv(file_path)
    
    def process_data(self) -> pd.DataFrame:
        ratings_df = self.load_dataset(self.config.ratings)
        logger.info(f"Extracting Movies features...⏳")
        movies_featues = self.movie_feature_extractor.generate_features(self.config.movies, self.config.tags)
        logger.info(f"Extracting Movies features completed ✅ ")
        logger.info(f"Extracting User features...⏳")
        users_feature = self.user_feature_extractor.generate_features(self.config.ratings, self.config.movies)
        logger.info(f"Extracting User features completed ✅")
        logger.info(f"Merging features with ratings...⏳")
        ratings_with_users = ratings_df.merge(users_feature, on='userId', how='left')
        # final_dataset = ratings_with_users.merge(movies_featues, on='userId', how='left')
        logger.info(f"Merging features with ratings completed ✅")

        return ratings_with_users

    def train_test_split(self, dataset: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
        pass
    

In [81]:
try:
    config = ConfigurationManager()
    get_data_processing_config = config.get_data_processing_config()
    data_preprocessor = DataPreprocessor(config=get_data_processing_config)
    dataset = data_preprocessor.process_data()

except Exception as e:
    logger.exception(f"Oops😟! An error occured: {e} ")
    raise e
dataset.head()

[2025-01-16 15:23:33,806: INFO: common: Yaml file : config/config.yaml loaded successfully]
[2025-01-16 15:23:33,834: INFO: common: Yaml file : params.yaml loaded successfully]
[2025-01-16 15:23:33,874: INFO: common: Yaml file : schema.yaml loaded successfully]
[2025-01-16 15:23:33,882: INFO: common: Created directory at: artifacts]
[2025-01-16 15:23:33,894: INFO: common: Created directory at: artifacts/data_transformation]
[2025-01-16 15:23:34,407: INFO: 3628626727: Extracting Movies features...⏳]
[2025-01-16 15:23:37,607: INFO: 3628626727: Extracting Movies features completed ✅ ]
[2025-01-16 15:23:37,615: INFO: 3628626727: Extracting User features...⏳]
[2025-01-16 15:23:44,919: INFO: 3628626727: Extracting User features completed ✅]
[2025-01-16 15:23:44,922: INFO: 3628626727: Merging features with ratings...⏳]


KeyboardInterrupt: 

: 

# User Features


## Step 1: Aggregate Ratings


In [23]:
# Aggregate user ratings
user_ratings = ratings_df.groupby('userId')['rating'].agg(['mean', 'count']).reset_index()
user_ratings.rename(columns={'mean': 'avg_rating', 'count': 'rating_count'}, inplace=True)

user_ratings.head()

,userId,avg_rating,rating_count
0,1,4.366379,232
1,2,3.948276,29
2,3,2.435897,39
3,4,3.555556,216
4,5,3.636364,44


## Step 2: User Preference for Genres


In [24]:
user_genres = ratings_df.merge(movies_df[['movieId'] + list(genre_set)], on='movieId', how='left')
user_genres.head()

,userId,movieId,rating,timestamp,Western,Romance,Fantasy,Adventure,Action,Film-Noir,...,Sci-Fi,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation
0,1,1,4.0,964982703,0,0,1,1,0,0,...,0,0,0,1,0,1,0,0,0,1
1,1,3,4.0,964981247,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,1,6,4.0,964982224,0,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
3,1,47,5.0,964983815,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0
4,1,50,5.0,964982931,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0


In [25]:
user_genre_prefenrences = user_genres.groupby('userId')[list(genre_set)].mean().reset_index()
user_genre_prefenrences.head()

,userId,Western,Romance,Fantasy,Adventure,Action,Film-Noir,(no genres listed),IMAX,War,...,Sci-Fi,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation
0,1,0.030172,0.112069,0.202586,0.366379,0.387931,0.004310,0.0,0.000000,0.094828,...,0.172414,0.077586,0.237069,0.357759,0.073276,0.181034,0.094828,0.293103,0.000000,0.125000
1,2,0.034483,0.034483,0.000000,0.103448,0.379310,0.000000,0.0,0.137931,0.034483,...,0.137931,0.068966,0.344828,0.241379,0.034483,0.000000,0.000000,0.586207,0.103448,0.000000
2,3,0.000000,0.128205,0.102564,0.282051,0.358974,0.000000,0.0,0.000000,0.128205,...,0.384615,0.025641,0.179487,0.230769,0.205128,0.128205,0.025641,0.410256,0.000000,0.102564
3,4,0.046296,0.268519,0.087963,0.134259,0.115741,0.018519,0.0,0.004630,0.032407,...,0.055556,0.106481,0.175926,0.481481,0.018519,0.046296,0.074074,0.555556,0.009259,0.027778
4,5,0.045455,0.250000,0.159091,0.181818,0.204545,0.000000,0.0,0.068182,0.068182,...,0.045455,0.022727,0.204545,0.340909,0.022727,0.204545,0.113636,0.568182,0.000000,0.136364


In [26]:
user_features = user_ratings.merge(user_genre_prefenrences, on='userId', how='left')
print(user_features.count())
user_features.head()

userId                610
avg_rating            610
rating_count          610
Western               610
Romance               610
Fantasy               610
Adventure             610
Action                610
Film-Noir             610
(no genres listed)    610
IMAX                  610
War                   610
Crime                 610
Sci-Fi                610
Mystery               610
Thriller              610
Comedy                610
Horror                610
Children              610
Musical               610
Drama                 610
Documentary           610
Animation             610
dtype: int64


,userId,avg_rating,rating_count,Western,Romance,Fantasy,Adventure,Action,Film-Noir,(no genres listed),...,Sci-Fi,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation
0,1,4.366379,232,0.030172,0.112069,0.202586,0.366379,0.387931,0.004310,0.0,...,0.172414,0.077586,0.237069,0.357759,0.073276,0.181034,0.094828,0.293103,0.000000,0.125000
1,2,3.948276,29,0.034483,0.034483,0.000000,0.103448,0.379310,0.000000,0.0,...,0.137931,0.068966,0.344828,0.241379,0.034483,0.000000,0.000000,0.586207,0.103448,0.000000
2,3,2.435897,39,0.000000,0.128205,0.102564,0.282051,0.358974,0.000000,0.0,...,0.384615,0.025641,0.179487,0.230769,0.205128,0.128205,0.025641,0.410256,0.000000,0.102564
3,4,3.555556,216,0.046296,0.268519,0.087963,0.134259,0.115741,0.018519,0.0,...,0.055556,0.106481,0.175926,0.481481,0.018519,0.046296,0.074074,0.555556,0.009259,0.027778
4,5,3.636364,44,0.045455,0.250000,0.159091,0.181818,0.204545,0.000000,0.0,...,0.045455,0.022727,0.204545,0.340909,0.022727,0.204545,0.113636,0.568182,0.000000,0.136364


# implamentation


In [27]:
import os
from mlProject import logger
from sklearn.model_selection import train_test_split
import pandas as pd

In [28]:
class DataTransformation:

    def __init__(self, config: DataTransformationConfig) -> None:
        
        self.config = config

    def train_test_spliting(self):

        data = pd.read_csv(self.config.data_path)
        train,test = train_test_split(data)

        train.to_csv(os.path.join(self.config.root_dir, "train.csv"), index=False)
        test.to_csv(os.path.join(self.config.root_dir, "test.csv"), index=False)

        logger.info("Data splitted into test and training set")
        logger.info(train.shape)
        logger.info(test.shape)

In [29]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.train_test_spliting()

except Exception as e:
    raise e

[2021-01-01 12:55:06,840: INFO: common: Yaml file : config/config.yaml loaded successfully]
[2021-01-01 12:55:06,850: INFO: common: Yaml file : params.yaml loaded successfully]
[2021-01-01 12:55:06,867: INFO: common: Yaml file : schema.yaml loaded successfully]
[2021-01-01 12:55:06,874: INFO: common: Created directory at: artifacts]


AttributeError: 'ConfigurationManager' object has no attribute 'get_data_transformation_config'

In [ ]:
# scale training data
item_train_unscaled = item_train
user_train_unscaled = user_train
y_train_unscaled    = y_train

scalerItem = StandardScaler()
scalerItem.fit(item_train)
item_train = scalerItem.transform(item_train)

scalerUser = StandardScaler()
scalerUser.fit(user_train)
user_train = scalerUser.transform(user_train)

scalerTarget = MinMaxScaler((-1, 1))
scalerTarget.fit(y_train.reshape(-1, 1))
y_train = scalerTarget.transform(y_train.reshape(-1, 1))
#ynorm_test = scalerTarget.transform(y_test.reshape(-1, 1))

print(np.allclose(item_train_unscaled, scalerItem.inverse_transform(item_train)))
print(np.allclose(user_train_unscaled, scalerUser.inverse_transform(user_train)))

In [ ]:
# GRADED_CELL
# UNQ_C1

num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  
  
  
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  
  
  
    ### END CODE HERE ###  
])

# create the user input and point to the base network
input_user = tf.keras.layers.Input(shape=(num_user_features))
vu = user_NN(input_user)
vu = tf.linalg.l2_normalize(vu, axis=1)

# create the item input and point to the base network
input_item = tf.keras.layers.Input(shape=(num_item_features))
vm = item_NN(input_item)
vm = tf.linalg.l2_normalize(vm, axis=1)

# compute the dot product of the two vectors vu and vm
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# specify the inputs and output of the model
model = tf.keras.Model([input_user, input_item], output)

model.summary()

In [ ]:
# model

from tensorflow.keras import layers, Model
import tensorflow as tf
import numpy as np
from typing import Tuple, List

class RecommenderNet(Model):
    def __init__(self, user_shape: int, movie_shape: int):
        super(RecommenderNet, self).__init__()
        
        # User tower layers
        self.user_input = layers.Input(shape=(user_shape,), name="user_input")
        self.user_dense1 = layers.Dense(64, activation="relu")
        self.user_dense2 = layers.Dense(32, activation="relu")
        
        # Movie tower layers
        self.movie_input = layers.Input(shape=(movie_shape,), name="movie_input")
        self.movie_dense1 = layers.Dense(64, activation="relu")
        self.movie_dense2 = layers.Dense(32, activation="relu")
        
        # Combined layers
        self.combined_dense = layers.Dense(64, activation="relu")
        self.output_layer = layers.Dense(1, activation="linear", name="output")
        
    def call(self, inputs):
        user_input, movie_input = inputs
        
        # User tower
        x1 = self.user_dense1(user_input)
        x1 = self.user_dense2(x1)
        
        # Movie tower
        x2 = self.movie_dense1(movie_input)
        x2 = self.movie_dense2(x2)
        
        # Combine towers
        combined = layers.concatenate([x1, x2])
        combined = self.combined_dense(combined)
        
        return self.output_layer(combined)
    
    def build_graph(self):
        """Create model graph for visualization"""
        model = Model(
            inputs=[self.user_input, self.movie_input],
            outputs=self.call([self.user_input, self.movie_input])
        )
        return model

class RecommenderTrainer:
    def __init__(
        self,
        user_shape: int,
        movie_shape: int,
        learning_rate: float = 0.001,
        batch_size: int = 32,
        epochs: int = 10
    ):
        self.model = RecommenderNet(user_shape, movie_shape)
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.history = None
        
    def compile_model(self):
        """Compile the model with specified parameters"""
        optimizer = tf.keras.optimizers.Adam(learning_rate=self.learning_rate)
        self.model.compile(
            optimizer=optimizer,
            loss='mse',
            metrics=['mae']
        )
        
    def train(
        self,
        X_user: np.ndarray,
        X_movie: np.ndarray,
        y: np.ndarray,
        validation_split: float = 0.2,
        callbacks: List = None
    ):
        """Train the model"""
        self.compile_model()
        self.history = self.model.fit(
            [X_user, X_movie],
            y,
            batch_size=self.batch_size,
            epochs=self.epochs,
            validation_split=validation_split,
            callbacks=callbacks
        )
        return self.history
    
    def predict(self, X_user: np.ndarray, X_movie: np.ndarray) -> np.ndarray:
        """Make predictions"""
        return self.model.predict([X_user, X_movie])
    
    def evaluate(
        self,
        X_user: np.ndarray,
        X_movie: np.ndarray,
        y: np.ndarray
    ) -> Tuple[float, float]:
        """Evaluate the model"""
        return self.model.evaluate([X_user, X_movie], y)
    
    def save_model(self, path: str):
        """Save the model"""
        self.model.save(path)
    
    @staticmethod
    def load_model(path: str):
        """Load a saved model"""
        return tf.keras.models.load_model(path)

# Usage example:
"""
# Initialize trainer
trainer = RecommenderTrainer(
    user_shape=X_user_np.shape[1],
    movie_shape=X_movie_np.shape[1],
    learning_rate=0.001,
    batch_size=32,
    epochs=10
)

# Define callbacks if needed
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )
]

# Train the model
history = trainer.train(
    X_user_np,
    X_movie_np,
    y,
    validation_split=0.2,
    callbacks=callbacks
)

# Make predictions
predictions = trainer.predict(X_user_test, X_movie_test)

# Evaluate model
loss, mae = trainer.evaluate(X_user_test, X_movie_test, y_test)

# Save model
trainer.save_model('recommender_model.h5')

# Load model later
loaded_model = RecommenderTrainer.load_model('recommender_model.h5')
"""